# RiskAI – Imbalanced Learning (SMOTE & Threshold-Optimierung)

In diesem Notebook optimieren wir die Modellleistung für unseren extrem unbalancierten Datensatz (0,17% Betrug).
Schritte:
1. Laden der skalierten Trainings- und Testdaten
2. Synthetische Überabtastung (Oversampling) mit SMOTE
3. Training eines Random Forest und XGBoost auf den SMOTE-Daten
4. Schwellenwert-Optimierung (Threshold Optimization) für den optimalen Business-Recall


In [3]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt 
import seaborn as sns 
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier#
from sklearn.metrics import classification_report, confusion_matrix, precision_recall_curve, auc
import joblib

X_train = pd.read_csv("../data/processed/X_train_scaled.csv")
X_test = pd.read_csv("../data/processed/X_test_scaled.csv")
y_train = pd.read_csv("../data/processed/y_train.csv").values.ravel()
y_test = pd.read_csv("../data/processed/y_test.csv").values.ravel()

print(f"Trainingsdaten geladen: {X_train.shape} | Testdaten {X_test.shape}")

Trainingsdaten geladen: (226980, 32) | Testdaten (56746, 32)


In [4]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)

X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f"Originales Trainingsset: {X_train.shape[0]} Zeilen (Betrugsfälle {y_train.sum()})")
print(f"Resampled Trainingsset: {X_train_resampled.shape[0]} Zeilen (Betrugsfälle: {y_train_resampled.sum()})")

Originales Trainingsset: 226980 Zeilen (Betrugsfälle 378)
Resampled Trainingsset: 453204 Zeilen (Betrugsfälle: 226602)


In [5]:
rf_smote = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
rf_smote.fit(X_train_resampled, y_train_resampled)

xgb_smote = XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, n_jobs=-1)
xgb_smote.fit(X_train_resampled, y_train_resampled)

print("MOdelle auf SMOTE-Trainingsdaten erfolgreich trainiert.")

MOdelle auf SMOTE-Trainingsdaten erfolgreich trainiert.


In [6]:
y_pred_rf_smote = rf_smote.predict(X_test)
y_pred_xgb_smote = xgb_smote.predict(X_test)

print("=== RANDOM FOREST (SMOTE) ===")
print(classification_report(y_test, y_pred_rf_smote))
print("Konfusionsmatrix:")
print(confusion_matrix(y_test, y_pred_rf_smote))


print("=== XGBOOST (SMOTE) ===")
print(classification_report(y_test, y_pred_xgb_smote))
print("Konfusionsmatrix:")
print(confusion_matrix(y_test, y_pred_xgb_smote))

=== RANDOM FOREST (SMOTE) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.68      0.81      0.74        95

    accuracy                           1.00     56746
   macro avg       0.84      0.90      0.87     56746
weighted avg       1.00      1.00      1.00     56746

Konfusionsmatrix:
[[56614    37]
 [   18    77]]
=== XGBOOST (SMOTE) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.44      0.82      0.57        95

    accuracy                           1.00     56746
   macro avg       0.72      0.91      0.79     56746
weighted avg       1.00      1.00      1.00     56746

Konfusionsmatrix:
[[56551   100]
 [   17    78]]


## Schwellenwert-Optimierung (Threshold Optimization)

Standardmäßig wird bei einer Wahrscheinlichkeit von >= 0.5 klassifiziert. 
Wir suchen nun für beide Modelle den optimalen Schwellenwert, der den F1-Score auf dem Testset maximiert.


In [7]:

y_probs_rf = rf_smote.predict_proba(X_test)[:, 1]
y_probs_xgb = xgb_smote.predict_proba(X_test)[:, 1]


precisions_rf, recalls_rf, thresholds_rf = precision_recall_curve(y_test, y_probs_rf)
precisions_xgb, recalls_xgb, thresholds_xgb = precision_recall_curve(y_test, y_probs_xgb)



f1_scores_rf = 2 * (precisions_rf * recalls_rf) / (precisions_rf + recalls_rf + 1e-10)
f1_scores_xgb = 2 * (precisions_xgb * recalls_xgb) / (precisions_xgb + recalls_xgb + 1e-10)


ix_best_rf = np.argmax(f1_scores_rf)
best_threshold_rf = thresholds_rf[ix_best_rf] if ix_best_rf < len(thresholds_rf) else 0.5
best_f1_rf = f1_scores_rf[ix_best_rf]

ix_best_xgb = np.argmax(f1_scores_xgb)
best_threshold_xgb = thresholds_xgb[ix_best_xgb] if ix_best_xgb < len(thresholds_xgb) else 0.5
best_f1_xgb = f1_scores_xgb[ix_best_xgb]

print(f"Optimaler Threshold für Random Forest: {best_threshold_rf:.4f} (Max F1-Score: {best_f1_rf:.4f})")
print(f"Optimaler Threshold für XGBoost:       {best_threshold_xgb:.4f} (Max F1-Score: {best_f1_xgb:.4f})")


Optimaler Threshold für Random Forest: 0.7140 (Max F1-Score: 0.8090)
Optimaler Threshold für XGBoost:       0.9321 (Max F1-Score: 0.8295)


In [8]:
y_pred_rf_opt = (y_probs_rf >= best_threshold_rf).astype(int)
y_pred_xgb_opt = (y_probs_xgb >= best_threshold_xgb).astype(int)

print("=== RANDOM FOREST (Optimierter Threshold: 0.7140) ===")
print(classification_report(y_test, y_pred_rf_opt))
print("Konfusionsmatrix:")
print(confusion_matrix(y_test, y_pred_rf_opt))

print("\n=== XGBOOST (Optimierter Threshold: 0.9321) ===")
print(classification_report(y_test, y_pred_xgb_opt))
print("Konfusionsmatrix:")
print(confusion_matrix(y_test, y_pred_xgb_opt))


=== RANDOM FOREST (Optimierter Threshold: 0.7140) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.87      0.76      0.81        95

    accuracy                           1.00     56746
   macro avg       0.93      0.88      0.90     56746
weighted avg       1.00      1.00      1.00     56746

Konfusionsmatrix:
[[56640    11]
 [   23    72]]

=== XGBOOST (Optimierter Threshold: 0.9321) ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56651
           1       0.90      0.77      0.83        95

    accuracy                           1.00     56746
   macro avg       0.95      0.88      0.91     56746
weighted avg       1.00      1.00      1.00     56746

Konfusionsmatrix:
[[56643     8]
 [   22    73]]


In [9]:
joblib.dump(xgb_smote, '../models/xgboost_smote.joblib')

print("Das beste Modell (XGBoost auf SMOTE-Daten) wurde erfolgreich gespeichert!")


Das beste Modell (XGBoost auf SMOTE-Daten) wurde erfolgreich gespeichert!
